# Energy Assistant — Diagnostics Notebook

Two independent diagnostic scripts cover the full picture of the running system.

| Section | Script | What it does |
|---------|--------|--------------|
| **Simulation** | `diagnostics_sim.py` | Fetches live data, runs MILP, shows energy-flow chart and cost savings |
| **Live status** | `diagnostics_live.py` | Connects to the running server, shows setpoints, ledger and plan chart |

Run cells in order.  The **Live** section requires the server to be running:
```
python -m energy_assistant
```


In [ ]:
import subprocess
import sys
from pathlib import Path

from IPython.display import HTML, display

# Resolve repo root from the notebook's location.
try:
    _nb_path = Path(globals().get("__vsc_ipynb_file__") or __file__).resolve()
except NameError:
    _nb_path = Path().resolve() / "visualize_optimizer.ipynb"

_nb_dir = _nb_path.parent
REPO_ROOT = _nb_dir.parent if _nb_dir.name == "notebooks" else _nb_dir

# Output paths for saved charts.
SIM_CHART  = Path("/tmp/ea_sim_chart.html")
LIVE_CHART = Path("/tmp/ea_live_plan.html")

# Server URL for the live diagnostics section.
API_URL = "http://localhost:8088"

print(f"Repo root : {REPO_ROOT}")
print(f"API URL   : {API_URL}")

## Simulation Diagnostics

Fetches live prices, PV forecast and battery SoC from ioBroker, runs the MILP optimizer and shows:

- Battery overview (cost basis, terminal value, discharge threshold)
- Charge / discharge totals per battery
- Cost comparison: optimised vs no-battery baseline
- Five-panel interactive Plotly chart: energy flows · forecasts · per-step saving · cumulative saving · SoC

In [ ]:
if SIM_CHART.exists():
    SIM_CHART.unlink()

result = subprocess.run(
    [
        sys.executable,
        str(REPO_ROOT / "scripts" / "diagnostics_sim.py"),
        "--save-html", str(SIM_CHART),
    ],
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.returncode != 0:
    print("STDERR:\n", result.stderr)

if result.returncode == 0 and SIM_CHART.exists():
    display(HTML(SIM_CHART.read_text()))

## Live Diagnostics

Connects to the running server via the REST API and shows:

- Current grid power, spot price and export price
- Per-device power and SoC
- Active control setpoints
- Battery ledger (stored energy + cost basis)
- Five-panel interactive Plotly chart: energy flows · forecasts · per-step saving · cumulative saving · SoC

> **Requires** the server to be running. Override the URL by changing `API_URL` in the setup cell above.

In [ ]:
if LIVE_CHART.exists():
    LIVE_CHART.unlink()

result = subprocess.run(
    [
        sys.executable,
        str(REPO_ROOT / "scripts" / "diagnostics_live.py"),
        "--url", API_URL,
        "--save-html", str(LIVE_CHART),
    ],
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.returncode != 0:
    print("STDERR:\n", result.stderr)

if result.returncode == 0 and LIVE_CHART.exists():
    display(HTML(LIVE_CHART.read_text()))